# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [79]:
# importar librerías
import pandas as pd
import numpy as np
import unicodedata

In [80]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv') # tu código aquí
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv') # tu código aquí
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv') # tu código aquí

In [81]:
# explorar datasets
# tu código aquí
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [82]:
catalog.head()

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [83]:
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [84]:
# Vemos el tamaño de los 3 dataset
print("Orders shape:", orders.shape)
print("Catalog shape:", catalog.shape)
print("Marketing shape:", marketing.shape)

Orders shape: (25100, 12)
Catalog shape: (7, 4)
Marketing shape: (1620, 5)


In [85]:
# Revisamos la información general de cada dataset
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [86]:
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [87]:
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [88]:

# Revisión de valores únicos para detectar inconsistencias
print("Países únicos:", orders['pais'].unique())
print("Dispositivos únicos:", orders['dispositivo'].unique())
print("Categorías en orders:", orders['categoria_producto'].unique())
print("Categorías en catalog:", catalog['categoria_producto'].unique())

# Estandarización y limpieza de texto (minúsculas/mayúsculas y quitar espacios sobrantes)
for col in ['pais', 'dispositivo', 'fuente_referencia', 'categoria_producto']:
    if col in orders.columns:
        orders[col] = orders[col].astype(str).str.strip().str.title()

for col in ['categoria_producto']:
    if col in catalog.columns:
        catalog[col] = catalog[col].astype(str).str.strip().str.title()

Países únicos: ['Argentina' 'Mexico' 'Colombia' 'mexico' 'colombia' 'argentina' nan]
Dispositivos únicos: ['desktop' 'mobile' nan]
Categorías en orders: ['Moda' 'Electronica' 'Hogar' nan]
Categorías en catalog: ['Electrónica' 'Hogar' 'Moda']


In [89]:
# normalizar mayúsculas/minúsculas 
def limpiar_texto(texto):
    if pd.isna(texto):
        return texto

    texto = str(texto).strip().title()

    texto = ''.join(
        c for c in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(c)
    )

    return texto

In [90]:
# normalizar mayúsculas/minúsculas
orders['categoria_producto'] = orders['categoria_producto'].apply(limpiar_texto)
catalog['categoria_producto'] = catalog['categoria_producto'].apply(limpiar_texto)

In [91]:
# tu código aquí
#Convertir la fecha al tipo correcto
orders['fecha_hora_pedido'] = pd.to_datetime(
    orders['fecha_hora_pedido'],
    errors='coerce'
)

In [92]:
# Validar que cantidad y precios no sean ilógicos (<= 0)
orders = orders[orders['cantidad'] > 0]
orders = orders[orders['precio_unitario'] >= 0]
orders = orders[orders['monto_descuento'] >= 0]
orders = orders[orders['monto_total'] >= 0]

In [93]:
# Verificar la consistencia de monto_total con la fórmula teórica
monto_calculado = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']

# Evaluamos diferencia aceptando un pequeño margen por redondeo de decimales
inconsistencias = orders[abs(orders['monto_total'] - monto_calculado) > 0.01]

print(f"Cantidad de filas con montos inconsistentes: {len(inconsistencias)}")

# Si se encuentran inconsistencias, se ajusta el dataset conservando solo los coherentes:
orders = orders[abs(orders['monto_total'] - monto_calculado) <= 0.01]

Cantidad de filas con montos inconsistentes: 1146


In [94]:
# Eliminar filas duplicadas si las hubiera
orders = orders.drop_duplicates()

In [95]:
# Limpieza de 'catalog' ---
catalog = catalog.drop_duplicates()
catalog = catalog[catalog['costo_unitario'] >= 0]

In [96]:
# Limpieza de 'marketing' ---
marketing['fecha'] = pd.to_datetime(marketing['fecha'])
marketing = marketing.drop_duplicates()
marketing = marketing[marketing['gasto'] >= 0]

In [97]:
# Verificación de valores nulos finales
print("Nulos en Orders:\n", orders.isnull().sum())

Nulos en Orders:
 id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo            0
fuente_referencia      0
nombre_producto       30
categoria_producto     0
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
dtype: int64


In [98]:
# Tratamiento de nulos en Orders
# Opción A: Rellenar con 'Desconocido' / 'Sin definir' para variables categóricas
orders['pais'] = orders['pais'].fillna('Desconocido')
orders['dispositivo'] = orders['dispositivo'].fillna('Desconocido')
orders['fuente_referencia'] = orders['fuente_referencia'].fillna('Desconocido')

# Opción B: Si faltan datos clave del producto y no son recuperables, eliminar esas filas
orders = orders.dropna(subset=['nombre_producto', 'categoria_producto'])

# Verificar que ya no queden nulos críticos
print("Nulos restantes en Orders:\n", orders.isnull().sum())

Nulos restantes en Orders:
 id_pedido             0
id_usuario            0
fecha_hora_pedido     0
pais                  0
dispositivo           0
fuente_referencia     0
nombre_producto       0
categoria_producto    0
cantidad              0
precio_unitario       0
monto_descuento       0
monto_total           0
dtype: int64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [99]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [100]:
# tu código aquí
# Unimos las tablas para los calculos
orders_catalog = orders.merge(
    catalog[['nombre_producto', 'costo_unitario', 'proveedor']],
    on='nombre_producto',
    how='left'
)
orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21,Bowers LLC
2,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21,Bowers LLC
3,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64,Long-Reid
4,order_5,user_2792,2025-04-22,Mexico,Desktop,Organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34,25.21,Bowers LLC


In [101]:
orders_catalog.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 23774 entries, 0 to 23773
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           23774 non-null  object        
 1   id_usuario          23774 non-null  object        
 2   fecha_hora_pedido   23774 non-null  datetime64[ns]
 3   pais                23774 non-null  object        
 4   dispositivo         23774 non-null  object        
 5   fuente_referencia   23774 non-null  object        
 6   nombre_producto     23774 non-null  object        
 7   categoria_producto  23774 non-null  object        
 8   cantidad            23774 non-null  float64       
 9   precio_unitario     23774 non-null  float64       
 10  monto_descuento     23774 non-null  float64       
 11  monto_total         23774 non-null  float64       
 12  costo_unitario      23774 non-null  float64       
 13  proveedor           23774 non-null  object    

In [102]:
orders_catalog['costo_unitario'].isna().sum()

0

In [103]:
#Calcular el costo total de cada pedido
orders_catalog['costo_total'] = (
    orders_catalog['cantidad'] *
    orders_catalog['costo_unitario']
)
orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_total
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,189.31,Mcmillan-Rhodes,378.62
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21,Bowers LLC,25.21
2,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21,Bowers LLC,25.21
3,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64,Long-Reid,176.64
4,order_5,user_2792,2025-04-22,Mexico,Desktop,Organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34,25.21,Bowers LLC,25.21


In [104]:
#Calcular la ganancia (Profit)
orders_catalog['profit'] = (
    orders_catalog['monto_total'] -
    orders_catalog['costo_total']
)
orders_catalog.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_total,profit
0,order_0,user_6993,2025-05-22,Argentina,Desktop,Organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,189.31,Mcmillan-Rhodes,378.62,286.75
1,order_1,user_1329,2025-06-15,Mexico,Desktop,Paid_Search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,25.21,Bowers LLC,25.21,146.65
2,order_3,user_4510,2025-06-09,Colombia,Mobile,Social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,25.21,Bowers LLC,25.21,217.66
3,order_4,user_5044,2025-03-30,Argentina,Desktop,Paid_Search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,176.64,Long-Reid,176.64,159.64
4,order_5,user_2792,2025-04-22,Mexico,Desktop,Organic,Tablet-Standard-64GB,Electronica,1.0,179.34,0.0,179.34,25.21,Bowers LLC,25.21,154.13


In [105]:
# Calculamos la entabilidad del negocio
revenue_total = orders_catalog['monto_total'].sum()
print(f"Ingreso total (Revenue): ${revenue_total:,.2f}")

Ingreso total (Revenue): $51,570,780.18


In [106]:
# Calculamos el costo total
costo_total = orders_catalog['costo_total'].sum()
print(f"Costo total: ${costo_total:,.2f}")

Costo total: $42,879,517.63


In [107]:
# Calculamos la inversión total en marketing
marketing_total = marketing['gasto'].sum()
print(f"Inversión en marketing: ${marketing_total:,.2f}")

Inversión en marketing: $2,871,843.53


In [108]:
## Calculamos el Profit del negocio
profit_total = revenue_total - costo_total - marketing_total
print(f"Profit del negocio: ${profit_total:,.2f}")

Profit del negocio: $5,819,419.02


In [109]:
# ¿El negocio es rentable?
if profit_total > 0:
    print("El negocio es rentable.")
else:
    print("El negocio NO es rentable.")

El negocio es rentable.


In [110]:
# ¿Cuál es el ticket promedio por orden?
ticket_promedio = orders_catalog['monto_total'].mean()
print(f"Ticket promedio: ${ticket_promedio:,.2f}")

Ticket promedio: $2,169.21


In [111]:
#¿Cuál es la cantidad promedio de productos?
cantidad_promedio = orders_catalog['cantidad'].mean()
print(f"Cantidad promedio por pedido: {cantidad_promedio:.2f}")

Cantidad promedio por pedido: 7.37


In [112]:
# ¿Cuál es el producto más vendido?
producto_mas_vendido = (
    orders_catalog
    .groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)
print(producto_mas_vendido)

nombre_producto
Laptop-Gaming-16GB      143914.0
Vacuum-Pro-Black          5952.0
Blender-XL-Red            5911.0
Jacket-Winter-M           5840.0
Sneakers-Urban-42         5774.0
Phone-Pro-128GB           3906.0
Tablet-Standard-64GB      3901.0
Name: cantidad, dtype: float64


In [113]:
# ¿Cuánto se ha gastado en marketing por canal?
gasto_canal = (
    marketing
    .groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)
print(gasto_canal)

canal
social         918043.21
organic        913533.01
paid_search    863088.21
Name: gasto, dtype: float64


In [114]:
# Validamos los resultados
orders_catalog[
    [
        'nombre_producto',
        'cantidad',
        'monto_total',
        'costo_unitario',
        'costo_total',
        'profit'
    ]
].head(10)

,nombre_producto,cantidad,monto_total,costo_unitario,costo_total,profit
0,Jacket-Winter-M,2.0,665.37,189.31,378.62,286.75
1,Tablet-Standard-64GB,1.0,171.86,25.21,25.21,146.65
2,Tablet-Standard-64GB,1.0,242.87,25.21,25.21,217.66
3,Blender-XL-Red,1.0,336.28,176.64,176.64,159.64
4,Tablet-Standard-64GB,1.0,179.34,25.21,25.21,154.13
5,Blender-XL-Red,2.0,327.19,176.64,353.28,-26.09
6,Tablet-Standard-64GB,2.0,747.35,25.21,50.42,696.93
7,Laptop-Gaming-16GB,2.0,949.54,280.68,561.36,388.18
8,Sneakers-Urban-42,1.0,334.30,17.21,17.21,317.09
9,Phone-Pro-128GB,1.0,292.14,10.12,10.12,282.02


In [115]:
print("Filas en orders:", len(orders))
print("Filas en orders_catalog:", len(orders_catalog))

Filas en orders: 23774
Filas en orders_catalog: 23774


In [116]:
catalog['nombre_producto'].value_counts()

Blender-XL-Red          1
Laptop-Gaming-16GB      1
Phone-Pro-128GB         1
Jacket-Winter-M         1
Tablet-Standard-64GB    1
Sneakers-Urban-42       1
Vacuum-Pro-Black        1
Name: nombre_producto, dtype: int64

In [117]:
catalog[catalog['nombre_producto'].duplicated(keep=False)]

,nombre_producto,categoria_producto,costo_unitario,proveedor


In [118]:
orders_catalog[
    orders_catalog['nombre_producto'] == 'Laptop-Gaming-16GB'
]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_total,profit
7,order_8,user_2352,2025-03-01,Argentina,Desktop,Paid_Search,Laptop-Gaming-16GB,Electronica,2.0,477.27,5.0,949.54,280.68,"Fuller, Pena and Myers",561.36,388.18
15,order_16,user_3710,2025-02-11,Colombia,Mobile,Social,Laptop-Gaming-16GB,Electronica,2.0,355.07,10.0,700.14,280.68,"Fuller, Pena and Myers",561.36,138.78
39,order_43,user_6949,2025-02-12,Colombia,Nan,Social,Laptop-Gaming-16GB,Electronica,1.0,415.13,0.0,415.13,280.68,"Fuller, Pena and Myers",280.68,134.45
66,order_151,user_2376,2025-01-12,Nan,Desktop,Paid_Search,Laptop-Gaming-16GB,Electronica,1.0,369.86,10.0,359.86,280.68,"Fuller, Pena and Myers",280.68,79.18
87,order_172,user_1918,2025-01-14,Nan,Desktop,Organic,Laptop-Gaming-16GB,Electronica,1.0,490.44,10.0,480.44,280.68,"Fuller, Pena and Myers",280.68,199.76
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23708,order_24930,user_2192,2025-01-31,Colombia,Desktop,Paid_Search,Laptop-Gaming-16GB,Electronica,2.0,137.32,15.0,259.65,280.68,"Fuller, Pena and Myers",561.36,-301.71
23741,order_24964,user_5944,2025-01-30,Mexico,Mobile,Organic,Laptop-Gaming-16GB,Electronica,1.0,418.95,10.0,408.95,280.68,"Fuller, Pena and Myers",280.68,128.27
23745,order_24969,user_3409,2025-03-28,Argentina,Mobile,Organic,Laptop-Gaming-16GB,Electronica,2.0,84.53,0.0,169.05,280.68,"Fuller, Pena and Myers",561.36,-392.31
23763,order_24988,user_6007,2025-06-17,Mexico,Desktop,Social,Laptop-Gaming-16GB,Electronica,2.0,441.65,5.0,878.30,280.68,"Fuller, Pena and Myers",561.36,316.94


In [119]:
orders_catalog[
    orders_catalog['nombre_producto'] == 'Laptop-Gaming-16GB'
]['cantidad'].describe()

count     2636.000000
mean        54.595599
std        912.104853
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64

In [120]:
# Filtramos los valores atípicos usando un criterio estadístico para Laptop-Gaming-16GB
Q1 = orders_catalog['cantidad'].quantile(0.25)
Q3 = orders_catalog['cantidad'].quantile(0.75)
IQR = Q3 - Q1

limite_superior = Q3 + 1.5 * IQR

print(limite_superior)

3.5


In [121]:
# Identificamos los registros
orders_catalog[orders_catalog['cantidad'] > limite_superior]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,costo_unitario,proveedor,costo_total,profit
3272,order_3521,user_5812,2025-02-03,Mexico,Mobile,Paid_Search,Laptop-Gaming-16GB,Electronica,10000.0,43.14,0.0,431400.0,280.68,"Fuller, Pena and Myers",2806800.0,-2375400.0
3273,order_3522,user_3575,2025-03-29,Argentina,Desktop,Social,Laptop-Gaming-16GB,Electronica,10000.0,280.55,0.0,2805500.0,280.68,"Fuller, Pena and Myers",2806800.0,-1300.0
3334,order_3586,user_3380,2025-02-03,Mexico,Mobile,Paid_Search,Laptop-Gaming-16GB,Electronica,10000.0,490.35,0.0,4903500.0,280.68,"Fuller, Pena and Myers",2806800.0,2096700.0
3389,order_3643,user_4440,2025-01-07,Colombia,Desktop,Social,Laptop-Gaming-16GB,Electronica,10000.0,238.15,0.0,2381500.0,280.68,"Fuller, Pena and Myers",2806800.0,-425300.0
3401,order_3656,user_884,2025-01-01,Argentina,Mobile,Organic,Laptop-Gaming-16GB,Electronica,20000.0,297.66,0.0,5953200.0,280.68,"Fuller, Pena and Myers",5613600.0,339600.0
3412,order_3668,user_7270,2025-06-24,Mexico,Mobile,Paid_Search,Laptop-Gaming-16GB,Electronica,20000.0,348.31,0.0,6966200.0,280.68,"Fuller, Pena and Myers",5613600.0,1352600.0
3432,order_3689,user_6566,2025-06-16,Mexico,Desktop,Paid_Search,Laptop-Gaming-16GB,Electronica,10000.0,87.69,0.0,876900.0,280.68,"Fuller, Pena and Myers",2806800.0,-1929900.0
3465,order_3722,user_4723,2025-05-09,Argentina,Mobile,Paid_Search,Laptop-Gaming-16GB,Electronica,20000.0,442.01,0.0,8840200.0,280.68,"Fuller, Pena and Myers",5613600.0,3226600.0
3469,order_3726,user_2536,2025-02-12,Colombia,Desktop,Social,Laptop-Gaming-16GB,Electronica,20000.0,290.85,0.0,5817000.0,280.68,"Fuller, Pena and Myers",5613600.0,203400.0
3491,order_3748,user_7096,2025-02-23,Mexico,Desktop,Organic,Laptop-Gaming-16GB,Electronica,10000.0,336.93,0.0,3369300.0,280.68,"Fuller, Pena and Myers",2806800.0,562500.0


In [122]:
orders_catalog = orders_catalog[
    orders_catalog['cantidad'] <= limite_superior
]

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [123]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [124]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()


,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [125]:
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 8 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   id_usuario          120000 non-null  object
 1   id_sesion           120000 non-null  object
 2   nombre_evento       120000 non-null  object
 3   timestamp_evento    120000 non-null  object
 4   pais                120000 non-null  object
 5   dispositivo         120000 non-null  object
 6   fuente_referencia   120000 non-null  object
 7   categoria_producto  120000 non-null  object
dtypes: object(8)
memory usage: 7.3+ MB


In [126]:
events['nombre_evento'].value_counts()

first_visit         29957
add_to_cart         24157
select_item         23887
begin_checkout      17971
add_payment_info    12018
purchase            12010
Name: nombre_evento, dtype: int64

In [127]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS total_usuarios
FROM events
GROUP BY nombre_evento
ORDER BY total_usuarios DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,total_usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [128]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel_base AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS total_usuarios
    FROM events
    GROUP BY nombre_evento
),
funnel_metricas AS (
    SELECT 
        nombre_evento,
        total_usuarios,
        LAG(total_usuarios) OVER (ORDER BY total_usuarios DESC) AS usuarios_paso_anterior,
        FIRST_VALUE(total_usuarios) OVER (ORDER BY total_usuarios DESC) AS usuarios_iniciales
    FROM funnel_base
)
SELECT 
    nombre_evento,
    total_usuarios,
    ROUND(100.0 * total_usuarios / COALESCE(usuarios_paso_anterior, total_usuarios), 2) AS pct_conversion_paso,
    ROUND(100.0 * total_usuarios / usuarios_iniciales, 2) AS pct_conversion_total
FROM funnel_metricas
ORDER BY total_usuarios DESC;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,total_usuarios,pct_conversion_paso,pct_conversion_total
0,first_visit,7796,100.00,100.00
1,add_to_cart,7634,97.92,97.92
2,select_item,7582,99.32,97.26
3,begin_checkout,7208,95.07,92.46
4,add_payment_info,6250,86.71,80.17
5,purchase,6240,99.84,80.04


### Conclusiones e Interpretación del Funnel de Conversión

1. **Etapa de mayor caída (*Drop-off*):** La mayor pérdida de usuarios ocurre entre **`begin_checkout`** y **`add_payment_info`**, donde solo el **86.71%** de los usuarios avanza al registro de pago (pérdida del **13.29%** / 958 usuarios).
2. **Conversión Global:** La tasa de conversión total acumulada del funnel (de `first_visit` a `purchase`) es del **80.04%**, lo cual representa una retención general muy alta en el flujo de compra (6,240 compradores finales de 7,796 visitantes iniciales).
3. **Recomendaciones para Producto:**
   * Optimizar la pasarela de pago para reducir fricciones en el formulario de tarjeta.
   * Garantizar transparencia de costos (envío/impuestos) antes de que el usuario llegue a la pantalla de pago.
   * Habilitar métodos de pago alternativos (billeteras virtuales, pagos con un solo clic).

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [129]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [130]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity
LIMIT 10;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [131]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH user_cohorts AS (
    SELECT 
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_registro,
        TO_CHAR(CAST(fecha_registro AS DATE), 'YYYY-MM') AS cohorte_mes
    FROM users
),
activity_weeks AS (
    SELECT 
        u.cohorte_mes,
        u.id_usuario,
        (CAST(a.fecha_actividad AS DATE) - u.fecha_registro) AS dias_diferencia,
        a.activo
    FROM user_cohorts u
    LEFT JOIN user_activity a ON u.id_usuario = a.id_usuario
)
SELECT 
    cohorte_mes,
    COUNT(DISTINCT id_usuario) AS total_usuarios,
    COUNT(DISTINCT CASE WHEN dias_diferencia BETWEEN 1 AND 7 AND activo = 1 THEN id_usuario END) AS retenido_w1,
    COUNT(DISTINCT CASE WHEN dias_diferencia BETWEEN 8 AND 14 AND activo = 1 THEN id_usuario END) AS retenido_w2,
    COUNT(DISTINCT CASE WHEN dias_diferencia BETWEEN 15 AND 21 AND activo = 1 THEN id_usuario END) AS retenido_w3,
    ROUND(COUNT(DISTINCT CASE WHEN dias_diferencia BETWEEN 1 AND 7 AND activo = 1 THEN id_usuario END)::numeric / COUNT(DISTINCT id_usuario) * 100, 2) AS semana_1,
    ROUND(COUNT(DISTINCT CASE WHEN dias_diferencia BETWEEN 8 AND 14 AND activo = 1 THEN id_usuario END)::numeric / COUNT(DISTINCT id_usuario) * 100, 2) AS semana_2,
    ROUND(COUNT(DISTINCT CASE WHEN dias_diferencia BETWEEN 15 AND 21 AND activo = 1 THEN id_usuario END)::numeric / COUNT(DISTINCT id_usuario) * 100, 2) AS semana_3
FROM activity_weeks
GROUP BY cohorte_mes
ORDER BY cohorte_mes;

'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte_mes,total_usuarios,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,697,668,656,42.84,41.06,40.32
1,2025-02,1444,611,609,635,42.31,42.17,43.98
2,2025-03,1636,677,705,690,41.38,43.09,42.18
3,2025-04,1606,680,697,663,42.34,43.40,41.28
4,2025-05,1687,695,676,706,41.20,40.07,41.85


In [132]:
# Verificación rápida del % de actividad global
df_activity = pd.read_sql("SELECT * FROM user_activity", con=engine)
tasa_actividad_global = (df_activity['activo'] == 1).mean() * 100
print(f"Porcentaje de registros con activo = 1: {tasa_actividad_global:.2f}%")

Porcentaje de registros con activo = 1: 41.61%


### Conclusiones e Interpretación - Paso 4: Retención por Cohortes

### 1. Evaluación General de Retención
* La tasa de retención se mantiene en un rango promedio del **40% al 44%** a lo largo de las tres semanas analizadas ($W_1$, $W_2$ y $W_3$).
* Aunque una retención cercana al 40% en la Semana 3 es un indicador positivo de engagement en comparación con estándares de la industria, el comportamiento no muestra la curva descendente típica de desenganche progresivo. Los niveles de actividad permanecen notablemente estables entre semanas.

### 2. Comparación entre Cohortes
* **Estabilidad inter-mensual:** No se observan variaciones drásticas en la retención entre las cohortes analizadas (por ejemplo, la cohorte de enero presenta un comportamiento equivalente a la de mayo). 
* **Consistencia del perfil:** Esto indica que los usuarios adquiridos en distintos meses responden de manera idéntica a la plataforma, sin impactos visibles por estacionalidad o cambios recientes en la adquisición.

### 3. Recomendaciones de Negocio
1. **Profundizar en la definición de 'Usuario Activo':** Dado que la retención no disminuye con el tiempo, se recomienda revisar qué acciones específicas gatillan la marca `activo = 1` para diferenciar interacciones superficiales de uso con valor real.
2. **Estrategia de Fidelización:** Aprovechar la base estable de usuarios (~40%) para implementar programas de lealtad, venta cruzada (*cross-selling*) o suscripciones recurrentes, dado que el usuario que supera la primera semana demuestra una probabilidad alta de mantenerse activo.
3. **Optimización del Onboarding:** Enfocar los esfuerzos de marketing en convertir el ~56-60% de usuarios no activos durante los primeros 7 días mediante campañas automatizadas de reactivación.

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

### Definición de Hipótesis y Test Estadístico

Para evaluar si el rediseño de la interfaz de usuario (UI) en el checkout tuvo un impacto estadísticamente significativo en la conversión de compra, definimos el siguiente marco de evaluación:

- **$H_0$ (Hipótesis nula):** No existe diferencia significativa en la tasa de conversión entre el grupo de control (UI original) y el grupo de tratamiento (UI modificada) ($p_{\text{control}} = p_{\text{treatment}}$).
- **$H_1$ (Hipótesis alternativa):** Existe una diferencia significativa en la tasa de conversión entre ambos grupos ($p_{\text{control}} \neq p_{\text{treatment}}$).

**Test estadístico:** Prueba Z para la diferencia de dos proporciones independientes (Z-test for proportions).  
**Nivel de significancia ($\alpha$):** $0.05$ (5%).

In [133]:
# Carga de datos del experimento A/B
experiment = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

experiment

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12
...,...,...,...,...,...,...,...
9995,exp_user_9995,tratamiento,0,mobile,Mexico,99.61,2025-01-27
9996,exp_user_9996,tratamiento,0,desktop,Argentina,209.82,2025-01-03
9997,exp_user_9997,tratamiento,0,mobile,Argentina,271.15,2025-02-26
9998,exp_user_9998,control,0,mobile,Colombia,216.11,2025-02-03


In [134]:
# Vista rápida del conteo por variante
print("=== Conteo de usuarios por variante ===")
print(experiment['variante'].value_counts())

# Resumen de métricas de conversión usando las columnas 'variante' y 'convirtio'
metrics_summary = experiment.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean']).reset_index()
metrics_summary.columns = ['Variante', 'Total Usuarios', 'Conversiones', 'Tasa Conversión']
metrics_summary['Tasa Conversión (%)'] = (metrics_summary['Tasa Conversión'] * 100).round(2)

print("\n=== Resumen del Experimento ===")
print(metrics_summary)

=== Conteo de usuarios por variante ===
tratamiento    5035
control        4965
Name: variante, dtype: int64

=== Resumen del Experimento ===
      Variante  Total Usuarios  Conversiones  Tasa Conversión  \
0      control            4965           779         0.156898   
1  tratamiento            5035           820         0.162860   

   Tasa Conversión (%)  
0                15.69  
1                16.29  


In [135]:
# Prueba Z de proporciones
from statsmodels.stats.proportion import proportions_ztest

# Extraer el número de conversiones y total de usuarios por variante
count_conversions = metrics_summary['Conversiones'].values
n_obs = metrics_summary['Total Usuarios'].values

# Ejecución de la prueba Z (bilateral / two-sided)
z_stat, p_value = proportions_ztest(count=count_conversions, nobs=n_obs, alternative='two-sided')

# Resultados de la prueba
alpha = 0.05
print(f"Estadístico Z: {z_stat:.4f}")
print(f"p-value: {p_value:.5f}")
print(f"Nivel de significancia (alpha): {alpha}")

print("-" * 50)
if p_value < alpha:
    print("Resultado: Rechazamos la hipótesis nula (H0).")
    print("Conclusión: Existe una diferencia estadísticamente significativa en la tasa de conversión entre ambas variantes.")
else:
    print("Resultado: No se puede rechazar la hipótesis nula (H0).")
    print("Conclusión: No existe evidencia estadística suficiente para afirmar que el cambio en la UI afectó la tasa de conversión.")

Estadístico Z: -0.8133
p-value: 0.41606
Nivel de significancia (alpha): 0.05
--------------------------------------------------
Resultado: No se puede rechazar la hipótesis nula (H0).
Conclusión: No existe evidencia estadística suficiente para afirmar que el cambio en la UI afectó la tasa de conversión.


In [136]:
# Cálculo del Lift (Impacto relativo)

# Tasa de conversión extrayendo los valores de 'control' y 'tratamiento'
conv_control = metrics_summary.loc[metrics_summary['Variante'] == 'control', 'Tasa Conversión'].values[0]
conv_treatment = metrics_summary.loc[metrics_summary['Variante'] == 'tratamiento', 'Tasa Conversión'].values[0]

# Cálculo de diferencias
lift_absoluto = conv_treatment - conv_control
lift_relativo = (lift_absoluto / conv_control) * 100

print(f"Tasa de conversión Control: {conv_control * 100:.2f}%")
print(f"Tasa de conversión Tratamiento: {conv_treatment * 100:.2f}%")
print(f"Diferencia absoluta: {lift_absoluto * 100:.2f}% p.p.")
print(f"Impacto relativo (Lift): {lift_relativo:.2f}%")

Tasa de conversión Control: 15.69%
Tasa de conversión Tratamiento: 16.29%
Diferencia absoluta: 0.60% p.p.
Impacto relativo (Lift): 3.80%


### Conclusión Final del Experimento A/B

#### 1. Resumen de Hallazgos
* **Grupo Control (UI Original):** Tasa de conversión del **15.69%** (779 conversiones de 4,965 usuarios).
* **Grupo Tratamiento (UI Rediseñada):** Tasa de conversión del **16.29%** (820 conversiones de 5,035 usuarios).
* **Impacto Observado:** Un incremento relativo (*Lift*) del **+3.80%** (+0.60 puntos porcentuales) a favor del grupo de tratamiento.

#### 2. Evaluación Estadística (Prueba Z)
* **Estadístico Z:** -0.8133
* **p-value:** 0.4161 (41.61%)
* **Nivel de significancia ($\alpha$):** 0.05 (5%)

Dado que el $p\text{-value}$ ($0.4161$) es sustancialmente mayor que el nivel de significancia $\alpha$ ($0.05$), **no se puede rechazar la hipótesis nula ($H_0$)**. La diferencia del 0.60% observada en la tasa de conversión no es estadísticamente significativa y puede atribuirse únicamente al azar o ruido muestral.

#### 3. Recomendación de Negocio
* **No implementar el rediseño de UI de forma definitiva:** No existe evidencia técnica que justifique el despliegue de la nueva versión con el objetivo de aumentar la conversión.
* **Siguientes pasos:** Si el rediseño implica costos adicionales de mantenimiento o desarrollo, se recomienda mantener la versión actual o formular nuevas hipótesis de UX enfocadas en puntos de fricción más específicos del proceso de pago.

---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive